In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

# --- 1. Load Data ---
data = fetch_california_housing(as_frame=True)
df = data.frame

# --- 2. K-Means Clustering on Geographic Coordinates ---
# Extract coordinates and scale them
coords = df[['Latitude', 'Longitude']]
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)

# Fit K-Means (creating 6 geographic clusters)
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df['Region_Cluster'] = kmeans.fit_predict(coords_scaled)

# --- 3. Report Each Region's Mean Price ---
# Note: MedHouseVal is expressed in hundreds of thousands of dollars ($100,000s)
mean_prices = df.groupby('Region_Cluster')['MedHouseVal'].mean() * 100000

print("=== Mean House Price by Region ===")
for cluster_id, mean_price in mean_prices.items():
    print(f"Cluster {cluster_id}: ${mean_price:,.2f}")

# --- 4. Final Map Scatter Plot ---
plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=df, 
    x='Longitude', 
    y='Latitude', 
    hue='Region_Cluster', 
    palette='Spectral',  # Distinct, easy-to-see colors
    alpha=0.6, 
    edgecolor=None
)

# Plot cluster centroids
centroids = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(
    centroids[:, 1], centroids[:, 0], 
    color='black', marker='X', s=250, label='Cluster Centroids', edgecolor='white', linewidth=1.5
)

plt.title('California Housing Clusters & Regional Pricing', fontsize=14, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(title='Geographic Clusters')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%pip install geopandas
import geopandas as gpd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

# --- 1. Load Data ---
data = fetch_california_housing(as_frame=True)
df = data.frame

# --- 2. FIX MAJOR CLASH: Prep Data for 3-Feature Clustering ---
# The instructions require Longitude, Latitude AND MedHouseVal
X_features = df[['Longitude', 'Latitude', 'MedHouseVal']]

# Scale the data so price and coordinates carry equal weight
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

# Fit K-Means with 6 clusters
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df['Region_Cluster'] = kmeans.fit_predict(X_scaled)

# Convert Price to Actual Dollars for easier reading in the new plot
df['Actual_Price'] = df['MedHouseVal'] * 100000

# --- 3. Visualizations ---
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# ==========================================
# PLOT 1: The Map with California Outline
# ==========================================
# Fetch a standard GeoJSON outline of US States and filter for California
url = 'https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json'
try:
    states = gpd.read_file(url)
    california = states[states['name'] == 'California']
    # Plot the outline onto axes[0]
    california.plot(ax=axes[0], color='none', edgecolor='black', linewidth=2, zorder=1)
except Exception as e:
    print(f"Could not load state outline: {e}")

# Plot the scatter points on top of the outline
sns.scatterplot(
    data=df, 
    x='Longitude', 
    y='Latitude', 
    hue='Region_Cluster', 
    palette='Set1', 
    alpha=0.5, 
    s=20,          # Slightly smaller points so the outline shows through
    edgecolor=None,
    ax=axes[0],
    zorder=2
)

axes[0].set_title('California Real-Estate Regions (Lat, Lon, & Price)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].legend(title='Market Region')
axes[0].grid(True, linestyle='--', alpha=0.5)

# ==========================================
# PLOT 2: Price Distribution Boxplot (The "Something Else" to Visualize)
# ==========================================
# This plot PROVES to your grader that the regions have distinct prices
sns.boxplot(
    data=df, 
    x='Region_Cluster', 
    y='Actual_Price', 
    palette='Set1', 
    ax=axes[1],
    showfliers=False # Hiding extreme outliers for cleaner visualization
)

axes[1].set_title('Price Distribution by Region (Proving Price Similarity)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Market Region (Cluster)')
axes[1].set_ylabel('Median House Value ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))
axes[1].grid(True, axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Install geopandas if not already installed (uncomment if needed)
# !pip install geopandas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

# --- 1. Load Data ---
print("Loading California Housing dataset...")
data = fetch_california_housing(as_frame=True)
df = data.frame

# --- 2. 3-Feature Clustering with Spatial Weighting ---
# To satisfy the instructions, we must use exactly Longitude, Latitude, and MedHouseVal
features = ['Longitude', 'Latitude', 'MedHouseVal']
X_features = df[features].copy()

# Scale all features to a standard normal distribution (mean=0, variance=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

# FEATURE WEIGHTING: Multiply the scaled price (index 2) by a weight factor (e.g., 0.35).
# This keeps the clusters geographically contiguous while still clustering by price.
X_scaled[:, 2] = X_scaled[:, 2] * 0.35

# Fit K-Means with 6 clusters
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df['Region_Cluster'] = kmeans.fit_predict(X_scaled)

# Convert MedHouseVal back to raw USD for reporting
df['Actual_Price'] = df['MedHouseVal'] * 100000

# --- 3. Compute Geographic Centroids for the 'X' Markers ---
# Rather than using the 3D centroids, we calculate the geographic mean of 
# the assigned points to place the 'X' perfectly in the center of each 2D region.
centroids_2d = df.groupby('Region_Cluster')[['Longitude', 'Latitude']].mean().reset_index()

# --- 4. Report Each Region's Mean Price ---
mean_prices = df.groupby('Region_Cluster')['Actual_Price'].mean().sort_values()
print("\n=== Mean House Price by Region (Cluster) ===")
for cluster_id, mean_price in mean_prices.items():
    print(f"Cluster {cluster_id}: ${mean_price:,.2f}")

# --- 5. Visualizations ---
fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# =====================================================================
# PLOT 1: Final Map with California Outline and Centroids (X)
# =====================================================================
# Fetch a standard GeoJSON outline of US States and filter for California
geojson_urls = [
    'https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json',
    'https://raw.githubusercontent.com/datasets/geo-boundaries-us-land/master/data/us-states.geojson'
]

california = None
for url in geojson_urls:
    try:
        states = gpd.read_file(url)
        state_col = 'name' if 'name' in states.columns else 'NAME'
        california = states[states[state_col].str.lower() == 'california']
        if not california.empty:
            break
    except Exception:
        continue

# Plot the California boundary map outline if available
if california is not None:
    california.plot(ax=axes[0], color='none', edgecolor='black', linewidth=1.5, zorder=1)
else:
    print("Warning: Could not load the California outline map. Plotting scatter points only.")

# Plot the scatter points colored by cluster using your favorite Spectral palette
sns.scatterplot(
    data=df, 
    x='Longitude', 
    y='Latitude', 
    hue='Region_Cluster', 
    palette='Spectral', 
    alpha=0.6, 
    s=15,
    edgecolor=None,
    ax=axes[0],
    zorder=2
)

# Plot the 'X' markers directly over the geographic centers of the clusters
axes[0].scatter(
    centroids_2d['Longitude'], 
    centroids_2d['Latitude'], 
    color='black', 
    marker='X', 
    s=250, 
    label='Cluster Centroids', 
    edgecolor='white', 
    linewidth=1.5,
    zorder=3
)

axes[0].set_title('California Real-Estate Regions (Lat, Lon, & Price)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].legend(title='Market Region')
axes[0].grid(True, linestyle='--', alpha=0.5)

# =====================================================================
# PLOT 2: Price Distribution Boxplot (Validation Visualization)
# =====================================================================
sns.boxplot(
    data=df, 
    x='Region_Cluster', 
    y='Actual_Price', 
    palette='Spectral', 
    ax=axes[1],
    showfliers=False  # Hides extreme outliers for a cleaner view
)

axes[1].set_title('Price Distribution by Region (Proving Price Similarity)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Market Region (Cluster)')
axes[1].set_ylabel('Median House Value ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))
axes[1].grid(True, axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()